# Shared Train/Validation/Test Split for Predictive Models

In [14]:
from datetime import datetime

import polars as pl

from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
)

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The generated 70/15/15 train, validation and test files are shared inputs for all predictive models. The random split is grouped by calendar date, so all spatial units and time buckets of a day remain in the same split.

In [15]:
DATASETS = (
    PATHS.gold_1h_demand_hexagon,
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_2h_demand_hexagon,
    PATHS.gold_2h_demand_census_tracts,
    PATHS.gold_2h_demand_community_areas,
    PATHS.gold_4h_demand_hexagon,
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)
if MODEL_START_TS >= MODEL_END_TS:
    raise ValueError(
        f"MODEL_START_DATE must be before MODEL_END_DATE: "
        f"{MODEL_START_DATE} >= {MODEL_END_DATE}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
if RUN_MODE == "full":
    print(f"Model period: [{MODEL_START_DATE}, {MODEL_END_DATE})")
else:
    print("Model-period filter disabled in sample mode")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: full
Model period: [2025-01-01T00:00:00, 2026-05-01T00:00:00)
Inputs: ['GOLD_1H_DEMAND_HEXAGON_7.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_2H_DEMAND_HEXAGON_7.parquet', 'GOLD_2H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_HEXAGON_7.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet']
Output directory: /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data


In [12]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RUN_MODE == "full":
        df_split = df_split.filter(
            (pl.col("datetime_hour") >= MODEL_START_TS)
            & (pl.col("datetime_hour") < MODEL_END_TS)
        )

    if RANDOM:
        # Keep every spatial unit and time bucket from the same calendar day
        # in one split. Unique dates are ordered reproducibly by their hash.
        # Validation and test receive exactly the same number of complete days.
        date_assignment = (
            df_split
            .select(pl.col("datetime_hour").dt.date().alias("_split_date"))
            .unique()
            .with_columns(
                pl.col("_split_date").hash(seed=SEED).alias("_split_order")
            )
            .sort(["_split_order", "_split_date"])
            .collect()
            .with_row_index("_date_rank")
        )
        n_dates = date_assignment.height
        n_holdout_dates = round(n_dates * 0.15)
        n_train_dates = n_dates - 2 * n_holdout_dates
        if n_train_dates <= 0 or n_holdout_dates <= 0:
            raise ValueError(f"Not enough dates for grouped 70/15/15 split: {n_dates}")

        date_assignment = (
            date_assignment
            .with_columns(
                pl.when(pl.col("_date_rank") < n_train_dates)
                .then(pl.lit("train"))
                .when(pl.col("_date_rank") < n_train_dates + n_holdout_dates)
                .then(pl.lit("val"))
                .otherwise(pl.lit("test"))
                .alias("_split")
            )
            .select(["_split_date", "_split"])
        )
        bucketed = (
            df_split
            .with_columns(
                pl.col("datetime_hour").dt.date().alias("_split_date")
            )
            .join(date_assignment.lazy(), on="_split_date", how="inner")
        )
        train = bucketed.filter(pl.col("_split") == "train")
        val = bucketed.filter(pl.col("_split") == "val")
        test = bucketed.filter(pl.col("_split") == "test")
        helper_columns = ["_split_date", "_split"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    split_dates = {
        name: frame.select(
            pl.col("datetime_hour").dt.date().alias("date")
        ).unique().collect()
        for name, frame in {"train": train, "val": val, "test": test}.items()
    }
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = split_dates[left].join(split_dates[right], on="date", how="inner")
        if overlap.height:
            raise ValueError(f"Date leakage between {left} and {right}: {overlap.height} dates")
    date_counts = {name: dates.height for name, dates in split_dates.items()}

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {"counts": counts, "paths": output_paths}

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Grouped split dates: {date_counts}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON_7.parquet: {'total': 1769280, 'train': 1236672, 'val': 266304, 'test': 266304}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Written: {'train': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TRAIN.parquet'), 'val': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_VAL.parquet'), 'test': PosixPath('/Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TEST.parquet')}
GOLD_1H_DEMAND_CENSUS_TRACTS.parquet: {'total': 10219920, 'train': 7143408, 'val': 1538256, 'test': 1538256}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Written: {'t

In [13]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2026-03-23 20:00:00,3,1,20,0.866025,0.5,0.0,1.0,-0.866025,0.5,3.195,42.17,10.25,10.0,0.0,0,0,1,0,0,0,2026-03-23,0,2,72.0,4.0,45.0,1.0,4,4649,1162.25,398,2100,24.3,6.075,1.23,12.07,70.75,17.6875,6.5,30.75,7.81,1.9525,0.0,7.81,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79.06,19.765,6.5,39.06,"""Cash"""
2026-04-16 16:00:00,4,4,16,1.0,6.1232e-17,0.433884,-0.900969,-0.866025,-0.5,20.89,70.304,13.8,9.8,0.5,1,0,0,0,0,0,2026-04-16,0,2,72.0,4.0,45.0,1.0,12,18286,1523.833333,162,2551,73.36,6.113333,0.08,12.45,247.67,20.639167,4.5,37.0,15.55,1.295833,0.0,7.5,2.0,0.166667,0.0,2.0,1.0,0.083333,0.0,1.0,268.72,22.393333,5.5,45.0,"""Cash"""
2026-04-19 12:00:00,4,7,12,1.0,6.1232e-17,-0.781831,0.62349,1.2246e-16,-1.0,7.5,44.285,10.0,10.0,0.0,0,0,1,0,0,0,2026-04-19,0,2,72.0,4.0,45.0,1.0,19,30498,1605.157895,300,3600,115.81,6.095263,0.83,12.81,404.01,21.263684,5.5,34.72,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.157895,0.0,1.0,408.01,21.474211,5.5,35.22,"""Prcard"""
2026-04-19 00:00:00,4,7,0,1.0,6.1232e-17,-0.781831,0.62349,0.0,1.0,6.805,45.0275,13.75,10.0,0.0,0,1,0,0,0,0,2026-04-19,0,2,72.0,4.0,45.0,1.0,7,5181,740.142857,286,1367,31.42,4.488571,0.7,12.84,98.75,14.107143,5.0,32.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.285714,0.0,2.0,100.75,14.392857,5.0,32.75,"""Cash"""
2026-04-26 08:00:00,4,7,8,1.0,6.1232e-17,-0.781831,0.62349,0.866025,-0.5,8.332857,91.234286,5.285714,7.142857,0.0,0,0,0,1,0,0,2026-04-26,0,2,72.0,4.0,45.0,1.0,20,19485,974.25,169,2100,111.34,5.567,0.78,17.49,371.79,18.5895,5.25,51.56,18.15,0.9075,0.0,9.75,0.0,0.0,0.0,0.0,1.0,0.05,0.0,1.0,393.94,19.697,6.75,51.56,"""Cash"""
2026-04-21 04:00:00,4,2,4,1.0,6.1232e-17,0.781831,0.62349,0.866025,0.5,10.28,42.8475,8.25,10.0,0.0,1,0,0,0,0,0,2026-04-21,0,2,72.0,4.0,45.0,1.0,10,16747,1674.7,1236,2839,115.73,11.573,6.83,19.33,311.49,31.149,21.5,47.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,311.49,31.149,21.5,47.99,"""Prcard"""
2026-03-07 08:00:00,3,6,8,0.866025,0.5,-0.974928,-0.222521,0.866025,-0.5,16.667778,93.564444,8.444444,6.277778,11.1803,0,0,1,0,0,0,2026-03-07,0,2,72.0,4.0,45.0,1.0,19,19780,1041.052632,135,2691,87.17,4.587895,0.4,12.22,295.96,15.576842,4.5,32.5,5.57,0.293158,0.0,3.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,303.03,15.948947,4.5,32.5,"""Prcard"""
2026-04-25 16:00:00,4,6,16,1.0,6.1232e-17,-0.974928,-0.222521,-0.866025,-0.5,11.6675,66.1875,10.25,10.0,0.0,0,0,1,0,0,0,2026-04-25,0,2,72.0,4.0,45.0,1.0,14,21611,1543.642857,219,3716,99.77,7.126429,0.37,18.96,318.5,22.75,4.75,56.33,2.0,0.142857,0.0,2.0,0.0,0.0,0.0,0.0,1.0,0.071429,0.0,1.0,323.0,23.071429,4.75,56.33,"""Prcard"""
2026-03-07 16:00:00,3,6,16,0.866025,0.5,-0.974928,-0.222521,-0.866025,-0.5,12.29375,83.6775,16.75,9.0,0.0003,1,0,0,0,0,0,2026-03-07,0,2,72.0,4.0,45.0,1.0,12,14546,1212.166667,374,3222,64.72,5.393333,0.6,19.25,215.31,17.9425,6.25,57.53,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.083333,0.0,1.0,217.81,18.150833,6.25,57.53,"""Prcard"""
